# Lab 15: Q-Learning on FrozenLake

**Module 15** | Read `notes/15-rl-mdps-qlearning.pdf` before running this notebook.

Tabular Q-learning with epsilon-greedy exploration on FrozenLake-v1 and render a solved episode.

Run every cell top to bottom. Code is complete, no fill-in sections. Markdown cells explain what each block does and why.


## Runtime device

PyTorch can run on your NVIDIA GPU through CUDA or fall back to CPU. GPU execution moves matrix operations off the host and typically speeds up neural network training by a large factor. The next cell detects what is available and prints the active device so you know whether to expect fast training or should keep batch sizes small.


In [ ]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cuda":
    props = torch.cuda.get_device_properties(0)
    print(f"CUDA enabled, {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {props.total_memory / 1e9:.1f} GB")
else:
    print("CUDA not available, using CPU. Labs still run; training may take longer.")


## Tabular Q-learning on FrozenLake

FrozenLake-v1 is a 4x4 grid world: start at top-left, goal at bottom-right, holes terminate the episode. With `is_slippery=False` actions are deterministic, which makes tabular Q-learning reliable.

Q-learning updates action values from experience without a model of the environment. Epsilon-greedy exploration balances trying random moves (discovering the goal) with exploiting the current best-known actions.


### Environment and hyperparameters

There are 16 states and 4 actions (left, down, right, up). We train for 10,000 episodes with a fixed exploration rate. Printing hyperparameters up front makes it easy to reproduce or tune runs later.


In [ ]:
import gymnasium as gym
import numpy as np

env = gym.make("FrozenLake-v1", is_slippery=False)
nS = env.observation_space.n
nA = env.action_space.n
Q = np.zeros((nS, nA))

ALPHA = 0.8
GAMMA = 0.99
EPSILON = 0.1
N_EPISODES = 10_000
REPORT_EVERY = 2000

ACTION_NAMES = {0: "LEFT", 1: "DOWN", 2: "RIGHT", 3: "UP"}
ACTION_SYMBOLS = {0: "<-", 1: "v", 2: "->", 3: "^"}

print("Q-learning hyperparameters:")
print(f"  States (nS):        {nS}")
print(f"  Actions (nA):       {nA}")
print(f"  Learning rate:      {ALPHA}")
print(f"  Discount (gamma):   {GAMMA}")
print(f"  Exploration (eps):  {EPSILON}")
print(f"  Episodes:           {N_EPISODES:,}")
print(f"  Progress report:    every {REPORT_EVERY} episodes")
print()
print("Action mapping:", ACTION_NAMES)


### Q-learning loop

Each step applies the Bellman update: move the Q-value toward the observed reward plus discounted best next-state value. We track per-episode total reward and print a running success rate every 2,000 episodes (success means reaching the goal with reward 1.0).


In [ ]:
rewards_per_episode: list[float] = []

for episode in range(N_EPISODES):
    state, _ = env.reset()
    done = False
    total_reward = 0.0

    while not done:
        if np.random.random() < EPSILON:
            action = env.action_space.sample()
        else:
            action = int(np.argmax(Q[state]))

        next_state, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated
        total_reward += reward

        best_next = np.max(Q[next_state])
        Q[state, action] += ALPHA * (reward + GAMMA * best_next - Q[state, action])
        state = next_state

    rewards_per_episode.append(total_reward)

    if (episode + 1) % REPORT_EVERY == 0:
        window = rewards_per_episode[-REPORT_EVERY:]
        success_rate = 100.0 * np.mean(window)
        successes = int(np.sum(window))
        print(
            f"Episode {episode + 1:>5}/{N_EPISODES}: "
            f"success rate last {REPORT_EVERY} = {success_rate:5.1f}% "
            f"({successes}/{REPORT_EVERY} goals reached)"
        )

final_window = rewards_per_episode[-1000:]
final_success = 100.0 * np.mean(final_window)
print()
print(f"Final success rate (last 1000 episodes): {final_success:.1f}%")


### Q-table statistics

After training, summarize the learned value function. Non-zero entries indicate state-action pairs that received updates. States near the goal and along successful paths typically carry the highest Q-values.


In [ ]:
print("Q-table statistics:")
print(f"  Shape:           {Q.shape}")
print(f"  Min value:       {Q.min():.4f}")
print(f"  Max value:       {Q.max():.4f}")
print(f"  Mean value:      {Q.mean():.4f}")
print(f"  Non-zero entries: {np.count_nonzero(Q)} / {Q.size}")
print()

best_state_actions = []
for s in range(nS):
    best_a = int(np.argmax(Q[s]))
    best_state_actions.append((s, best_a, Q[s, best_a]))

top_states = sorted(best_state_actions, key=lambda x: x[2], reverse=True)[:5]
print("Top 5 states by best-action Q-value:")
for s, a, val in top_states:
    print(f"  state {s:2d}: best={ACTION_NAMES[a]:<6} Q={val:.4f}")


### Learned policy grid

The greedy policy picks `argmax_a Q(s, a)` per state. Holes and the goal are shown as-is from the map; arrows show the preferred move elsewhere.


In [ ]:
MAP_LAYOUT = [
    "SFFF",
    "FHFH",
    "FFFH",
    "HFFG",
]

policy = np.argmax(Q, axis=1)
print("Learned policy (4x4 grid, row-major):")
print("-" * 17)
for row_idx, row in enumerate(MAP_LAYOUT):
    cells = []
    for col_idx, ch in enumerate(row):
        state = row_idx * 4 + col_idx
        if ch in "SHG":
            cells.append(f" {ch} ")
        else:
            cells.append(f" {ACTION_SYMBOLS[policy[state]]} ")
    print("|".join(cells))
print("-" * 17)
print("S=start  G=goal  H=hole  arrows=greedy action")


### Render one solved episode

Replay the greedy policy from the start. Each step prints the action name, step reward, and cumulative reward. Text rendering works in notebooks without opening a separate window.


In [ ]:
render_env = gym.make("FrozenLake-v1", is_slippery=False, render_mode="ansi")
state, _ = render_env.reset()
cumulative = 0.0

print("Greedy episode replay:")
print(render_env.render())
print(f"Step 0: start state={state}, cumulative reward={cumulative:.1f}")
print()

steps = 0
done = False

while not done and steps < 20:
    action = int(np.argmax(Q[state]))
    state, reward, terminated, truncated, _ = render_env.step(action)
    done = terminated or truncated
    steps += 1
    cumulative += reward
    print(
        f"Step {steps}: action={ACTION_NAMES[action]:<5} "
        f"reward={reward:.1f} cumulative={cumulative:.1f}"
    )
    print(render_env.render())
    print()

if cumulative >= 1.0:
    print(f"Reached the goal in {steps} steps with cumulative reward {cumulative:.1f}.")
else:
    print(f"Episode ended in {steps} steps with cumulative reward {cumulative:.1f}.")

render_env.close()


### Evaluation

On the deterministic 4x4 map, a well-trained agent should reach success rates near 100% in the final episodes and navigate greedily to the goal without falling in holes. If success rate plateaus low, try more episodes, a higher `ALPHA`, or a larger `EPSILON` early in training (not changed here to keep the lab reproducible).


In [ ]:
print("Q-learning evaluation summary:")
print(f"  Episodes trained:     {N_EPISODES:,}")
print(f"  Final success rate:   {final_success:.1f}% (last 1000)")
print(f"  Q-table max value:    {Q.max():.4f}")
print(f"  Greedy episode steps: {steps}")
print(f"  Greedy cumulative:    {cumulative:.1f}")
print()
if final_success > 80.0 and cumulative >= 1.0:
    print("Result: policy learned successfully.")
else:
    print("Result: review hyperparameters or episode count if success remains low.")
